# Classical Neural Network Baseline

Notebook này xây dựng một bài toán **regression phi tuyến có cấu trúc Fourier**:

$$
y(x)=\sin(2x)+0.5\sin(6x)+0.25\cos(10x), \quad x \in [-\pi, \pi]
$$

Dữ liệu của thí nghiệm là một chiều nên baseline cổ điển hợp lý là một **MLP nhỏ** thay vì CNN. Mục tiêu không phải làm yếu mô hình classical, mà tạo một baseline có số trainable parameters gần với QNN Data Re-uploading để so sánh công bằng.


## 1. Imports

Notebook chỉ dùng `numpy` cho mô hình và tối ưu hóa, `matplotlib` cho trực quan hóa, và thư viện chuẩn Python để đo thời gian.


In [ ]:
import time
import copy
import numpy as np
import matplotlib.pyplot as plt


## 2. Configuration

Các hằng số này được giữ giống notebook QNN để hai mô hình dùng cùng dataset và cùng train/validation/test split.


In [ ]:
SEED = 42
N_SAMPLES = 400
TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
TEST_RATIO = 0.15
DOMAIN_MIN = -np.pi
DOMAIN_MAX = np.pi

rng = np.random.default_rng(SEED)

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25


## 3. Reproducibility và Dataset Generation

Dataset được sinh ngẫu nhiên đều trên miền $[-\pi, \pi]$ với `SEED = 42`. Split được tạo bằng `SEED + 1` và được in checksum để dễ kiểm tra hai notebook đang dùng cùng chỉ số train/validation/test.

Target không được chuẩn hóa trong thí nghiệm này. Cả MLP và QNN đều học trực tiếp cùng thang giá trị của hàm gốc.


In [ ]:
def target_function(x):
    return np.sin(2 * x) + 0.5 * np.sin(6 * x) + 0.25 * np.cos(10 * x)


def make_fourier_dataset(n_samples=N_SAMPLES, seed=SEED):
    data_rng = np.random.default_rng(seed)
    x = data_rng.uniform(DOMAIN_MIN, DOMAIN_MAX, size=n_samples)
    y = target_function(x)
    return x.reshape(-1, 1), y.reshape(-1, 1)


def make_split_indices(n_samples=N_SAMPLES, seed=SEED):
    split_rng = np.random.default_rng(seed + 1)
    indices = split_rng.permutation(n_samples)

    n_train = int(TRAIN_RATIO * n_samples)
    n_validation = int(VALIDATION_RATIO * n_samples)

    train_indices = indices[:n_train]
    validation_indices = indices[n_train:n_train + n_validation]
    test_indices = indices[n_train + n_validation:]

    return train_indices, validation_indices, test_indices


x_all, y_all = make_fourier_dataset()
train_indices, validation_indices, test_indices = make_split_indices()

x_train, y_train = x_all[train_indices], y_all[train_indices]
x_validation, y_validation = x_all[validation_indices], y_all[validation_indices]
x_test, y_test = x_all[test_indices], y_all[test_indices]

x_plot = np.linspace(DOMAIN_MIN, DOMAIN_MAX, 800).reshape(-1, 1)
y_plot = target_function(x_plot)

print('Dataset shape:', x_all.shape, y_all.shape)
print('Train / validation / test:', len(x_train), len(x_validation), len(x_test))
print('Split checksum:', int(train_indices.sum()), int(validation_indices.sum()), int(test_indices.sum()))


## 4. Visualization of Dataset

Hình dưới cho thấy target function thật cùng các điểm train, validation và test.


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(x_plot, y_plot, color='black', linewidth=2, label='Ground truth')
plt.scatter(x_train, y_train, s=16, alpha=0.65, label='train')
plt.scatter(x_validation, y_validation, s=20, alpha=0.75, label='validation')
plt.scatter(x_test, y_test, s=20, alpha=0.75, label='test')
plt.title('Fourier-structured regression dataset')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()


## 5. Model Definition

Baseline là MLP:

```text
Input scalar x
    |
Dense 1 -> 5
    |
Tanh
    |
Dense 5 -> 4
    |
Tanh
    |
Dense 4 -> 1
```

Số tham số:

- Layer 1: $(1 \times 5) + 5 = 10$
- Layer 2: $(5 \times 4) + 4 = 24$
- Output: $(4 \times 1) + 1 = 5$

Tổng cộng: **39 trainable parameters**, rất gần với QNN 3 qubit, 4 re-uploading layers và linear readout có **40 trainable parameters**.


In [ ]:
def initialize_mlp(random_generator, hidden_1=5, hidden_2=4):
    return {
        'W1': random_generator.normal(0, 0.4, size=(1, hidden_1)),
        'b1': np.zeros((1, hidden_1)),
        'W2': random_generator.normal(0, 0.4, size=(hidden_1, hidden_2)),
        'b2': np.zeros((1, hidden_2)),
        'W3': random_generator.normal(0, 0.4, size=(hidden_2, 1)),
        'b3': np.zeros((1, 1)),
    }


def mlp_forward(x, params):
    z1 = x @ params['W1'] + params['b1']
    a1 = np.tanh(z1)
    z2 = a1 @ params['W2'] + params['b2']
    a2 = np.tanh(z2)
    y_pred = a2 @ params['W3'] + params['b3']

    cache = {'x': x, 'z1': z1, 'a1': a1, 'z2': z2, 'a2': a2, 'y_pred': y_pred}
    return y_pred, cache


def mlp_backward(y_true, cache, params):
    n_samples = y_true.shape[0]
    d_y_pred = 2 * (cache['y_pred'] - y_true) / n_samples

    grads = {}
    grads['W3'] = cache['a2'].T @ d_y_pred
    grads['b3'] = np.sum(d_y_pred, axis=0, keepdims=True)

    d_a2 = d_y_pred @ params['W3'].T
    d_z2 = d_a2 * (1 - cache['a2'] ** 2)
    grads['W2'] = cache['a1'].T @ d_z2
    grads['b2'] = np.sum(d_z2, axis=0, keepdims=True)

    d_a1 = d_z2 @ params['W2'].T
    d_z1 = d_a1 * (1 - cache['a1'] ** 2)
    grads['W1'] = cache['x'].T @ d_z1
    grads['b1'] = np.sum(d_z1, axis=0, keepdims=True)

    return grads


def count_parameters(params):
    return int(sum(value.size for value in params.values()))


def predict_mlp(x, params):
    y_pred, _ = mlp_forward(x, params)
    return y_pred


classical_params = initialize_mlp(rng)
print('Classical trainable parameters:', count_parameters(classical_params))


## 6. Training

Mô hình dùng MSE loss và Adam optimizer. Hyperparameters được chọn theo validation loss, không dùng test set để tune.

Notebook lưu bản model tốt nhất theo validation MSE và dùng bản đó để đánh giá cuối cùng trên test set.


In [ ]:
def adam_update(params, grads, optimizer_state, learning_rate, beta1=0.9, beta2=0.999, epsilon=1e-8):
    optimizer_state['t'] += 1
    t = optimizer_state['t']

    for key in params:
        optimizer_state['m'][key] = beta1 * optimizer_state['m'][key] + (1 - beta1) * grads[key]
        optimizer_state['v'][key] = beta2 * optimizer_state['v'][key] + (1 - beta2) * (grads[key] ** 2)

        m_hat = optimizer_state['m'][key] / (1 - beta1 ** t)
        v_hat = optimizer_state['v'][key] / (1 - beta2 ** t)
        params[key] -= learning_rate * m_hat / (np.sqrt(v_hat) + epsilon)

    return params


def make_adam_state(params):
    return {
        't': 0,
        'm': {key: np.zeros_like(value) for key, value in params.items()},
        'v': {key: np.zeros_like(value) for key, value in params.items()},
    }


def iterate_minibatches(x, y, random_generator, batch_size):
    indices = random_generator.permutation(len(x))
    for start in range(0, len(x), batch_size):
        batch_indices = indices[start:start + batch_size]
        yield x[batch_indices], y[batch_indices]


def mean_squared_error(y_true, y_pred):
    return float(np.mean((y_true - y_pred) ** 2))


def train_mlp(params, epochs=250, batch_size=32, learning_rate=0.01):
    optimizer_state = make_adam_state(params)
    history = {'train_mse': [], 'validation_mse': []}
    best_params = copy.deepcopy(params)
    best_validation_mse = float('inf')
    start_time = time.perf_counter()

    for epoch in range(1, epochs + 1):
        for x_batch, y_batch in iterate_minibatches(x_train, y_train, rng, batch_size):
            _, cache = mlp_forward(x_batch, params)
            grads = mlp_backward(y_batch, cache, params)
            params = adam_update(params, grads, optimizer_state, learning_rate)

        train_pred = predict_mlp(x_train, params)
        validation_pred = predict_mlp(x_validation, params)
        train_mse = mean_squared_error(y_train, train_pred)
        validation_mse = mean_squared_error(y_validation, validation_pred)

        history['train_mse'].append(train_mse)
        history['validation_mse'].append(validation_mse)

        if validation_mse < best_validation_mse:
            best_validation_mse = validation_mse
            best_params = copy.deepcopy(params)

        if epoch == 1 or epoch % 25 == 0:
            print(f'Epoch {epoch:03d} | train MSE = {train_mse:.6f} | validation MSE = {validation_mse:.6f}')

    training_time = time.perf_counter() - start_time
    return best_params, history, best_validation_mse, training_time


best_classical_params, history, best_validation_mse, training_time = train_mlp(classical_params)


## 7. Evaluation Metrics

Regression được đánh giá bằng MSE, MAE và $R^2$ trên test set chưa dùng trong quá trình tune.


In [ ]:
def mean_squared_error(y_true, y_pred):
    return float(np.mean((y_true - y_pred) ** 2))


def mean_absolute_error(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))


def r2_score_numpy(y_true, y_pred):
    ss_residual = np.sum((y_true - y_pred) ** 2)
    ss_total = np.sum((y_true - np.mean(y_true)) ** 2)
    return float(1 - ss_residual / ss_total)


In [ ]:
test_predictions = predict_mlp(x_test, best_classical_params)
test_mse = mean_squared_error(y_test, test_predictions)
test_mae = mean_absolute_error(y_test, test_predictions)
test_r2 = r2_score_numpy(y_test, test_predictions)

print('------------------------------------')
print('Model: Classical Neural Network Baseline')
print(f'Trainable parameters: {count_parameters(best_classical_params)}')
print(f'Best validation MSE: {best_validation_mse:.6f}')
print(f'Test MSE: {test_mse:.6f}')
print(f'Test MAE: {test_mae:.6f}')
print(f'Test R^2: {test_r2:.6f}')
print(f'Training time: {training_time:.2f} seconds')
print('------------------------------------')


## 8. Visualization

Ba hình bắt buộc:

1. Ground Truth vs Prediction.
2. Training Curve.
3. Prediction Error.


In [ ]:
plot_predictions = predict_mlp(x_plot, best_classical_params)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(x_plot, y_plot, color='black', linewidth=2, label='Ground truth')
axes[0].plot(x_plot, plot_predictions, color='tab:blue', linewidth=2, label='Prediction')
axes[0].scatter(x_train, y_train, s=14, alpha=0.45, label='train')
axes[0].scatter(x_test, y_test, s=18, alpha=0.75, label='test')
axes[0].set_title('Classical Neural Network Regression')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].legend()

axes[1].plot(history['train_mse'], label='Train MSE', linewidth=2)
axes[1].plot(history['validation_mse'], label='Validation MSE', linewidth=2)
axes[1].set_title('Training Curve')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE')
axes[1].legend()

sorted_test = np.argsort(x_test[:, 0])
sorted_x_test = x_test[sorted_test]
sorted_abs_error = np.abs(y_test[sorted_test] - test_predictions[sorted_test])
axes[2].plot(sorted_x_test, sorted_abs_error, marker='o', linewidth=1.5)
axes[2].set_title('Prediction Error')
axes[2].set_xlabel('x')
axes[2].set_ylabel('|y_true - y_pred|')

plt.tight_layout()
plt.show()


## 9. Results Summary và Scientific Interpretation

Baseline classical dùng MLP nhỏ với 39 trainable parameters, gần bằng QNN 40 parameters. Mô hình này không bị cố tình làm yếu: nó dùng `Tanh`, Adam, mini-batch training, validation-based model selection và test set độc lập.

Khi phân tích kết quả sau khi chạy notebook, hãy kiểm tra:

- Đường prediction có bám được các dao động tần số cao của target function không.
- Train MSE và validation MSE có tách xa nhau không để nhận biết overfitting.
- Prediction error lớn nhất nằm ở vùng nào của miền $[-\pi, \pi]$.

Notebook này chỉ cung cấp baseline cổ điển cho cùng bài toán. Kết luận về QNN advantage chỉ được đưa ra sau khi so sánh metric thực nghiệm của cả hai notebook, không suy diễn trước từ kiến trúc mô hình.
